# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs with `mlcroissant`. This section gives a structural overview of the dataset using their `@id` values.

In [ ]:
# Show record set information
record_sets = dataset.record_sets
print("Number of record sets:", len(record_sets))

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields:")
    for fld in rs.fields:
        print(f"    - Field @id: {fld.id}, Name: {fld.name}, Data type: {fld.data_type}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. We will list all record set `@id`s and load all available data for further analysis.

In [ ]:
# List all record sets by @id
record_set_ids = [rs.id for rs in dataset.record_sets]
print("All record set @id values:", record_set_ids)

# Load each record set into a dataframe
dataframes = {}
for rs_id in record_set_ids:
    # Use the mlcroissant Dataset.records() function with record_set @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set @id: {rs_id} | Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Preview the first record set, if available
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set @id: {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing on a selected field of a selected record set. Use the available field and record_set `@id`s to ensure referencing is always by `@id`.

In [ ]:
import numpy as np
# For demonstration, select the first numeric field from the first record set (if present)
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    
    # Attempt to choose a numeric field based on dtype
    numeric_field_id = None
    for fld in dataset.record_sets[0].fields:
        # Try to find first numeric type (Integer or Float)
        if fld.data_type and ('Integer' in fld.data_type or 'Float' in fld.data_type or 'Number' in fld.data_type):
            if fld.id in df.columns:
                numeric_field_id = fld.id
                break
    
    if numeric_field_id is not None:
        print(f"Selected numeric field @id for analysis: {numeric_field_id}")
        # Convert column to numeric, coerce errors to NaN
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors="coerce")
        
        threshold = np.nanmean(df[numeric_field_id])  # use the mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in record set {rs_id} where {numeric_field_id} > {threshold:.2f} (sample):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to select a grouping field (first categorical/string field with less than half the row count in unique values)
        group_field = None
        for fld in dataset.record_sets[0].fields:
            if fld.data_type and ("Text" in fld.data_type or "String" in fld.data_type):
                if fld.id in filtered_df.columns:
                    nunique = filtered_df[fld.id].nunique(dropna=True)
                    if nunique > 1 and nunique < 0.5 * filtered_df.shape[0]:
                        group_field = fld.id
                        break
        if group_field:
            print(f"\nGrouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in the first record set.")
else:
    print("No record sets found in dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn for enhanced visualization. Example: histogram of numeric field, or bar plot if grouping variable exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Continue from previous EDA step, reuse df, numeric_field_id, group_field if available
if 'df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 5))
        sns.barplot(y=group_means.index, x=group_means.values, palette='viridis')
        plt.title(f"Mean of {numeric_field_id} by {group_field}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("Visualization skipped: No suitable numeric field found.")

## 6. Conclusion

- The FAIR^2 dataset was successfully loaded and explored using the `mlcroissant` library.
- Record sets, fields, and columns were referenced by their `@id` to ensure accurate mapping.
- Initial exploratory data analysis and visualization highlight the potential for analyzing predictors of adoption in rangeland management.
- Further analysis may explore deeper statistical relationships, missing data patterns, and more complex groupings based on available metadata.
